In [56]:
import pyspark
from pyspark.sql import SparkSession, functions as F
import os
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampNTZType, DoubleType, NullType, LongType

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/15 20:20:52 WARN Utils: Your hostname, DESKTOP-3J23N5C, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/15 20:20:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 20:20:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# for y in [2020,2021]:
#         for i in range(1,13):
#             df_green = spark.read \
#                     .parquet(f"data/raw/green/{y}/{i:02d}")
#             df_green.repartition(4) \
#                 .write.parquet(f"data/pq/green/{y}/{i:02d}")
        
        

In [ ]:
# for y in [2020,2021]:
#         for i in range(1,13):
#             df_yellow = spark.read \
#                     .parquet(f"data/raw/yellow/{y}/{i:02d}")
#             df_yellow.repartition(4) \
#                 .write.parquet(f"data/pq/yellow/{y}/{i:02d}")
        

In [82]:
df_green = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet('data/pq/green/')

In [70]:
df_green.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: void (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [83]:
df_yellow = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet('data/pq/yellow/')

In [85]:
df_yellow.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: void (nullable = true)



In [73]:
len(set(df_green.columns) & set(df_yellow.columns)) # perform intersection of columns to findout whcih ones to keep

16

In [86]:
df_green = df_green.withColumnsRenamed({'tpep_pickup_datetime':'pickup_datetime', 'tpep_dropoff_datetime':'dropoff_datetime'})
df_yellow = df_yellow.withColumnsRenamed({'tpep_pickup_datetime':'pickup_datetime', 'tpep_dropoff_datetime':'dropoff_datetime'})

In [87]:
common_columns = [column for column in df_green.columns if column in set(df_yellow.columns)]
len(common_columns)

16

In [88]:
df_green_comb = df_green \
                        .select(common_columns) \
                        .withColumn('service_type', F.lit('green'))

In [89]:
df_yellow_comb = df_yellow \
                        .select(common_columns) \
                        .withColumn('service_type', F.lit('yellow'))

In [90]:
df_trips_data = df_green_comb.unionAll(df_yellow_comb)

In [ ]:
df_trips_data.groupBy('service_type').count().show()

In [91]:
df_trips_data.createOrReplaceTempView('trips_data')  # register this DF as a temp table so we can use spark sql to query it

In [92]:
spark.sql("""
    SELECT * from trips_data
    LIMIT 10;
""").show()

+--------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------------------+------------+------------+--------------------+------------+
|VendorID|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|payment_type|congestion_surcharge|service_type|
+--------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------------------+------------+------------+--------------------+------------+
|       2|                 N|       1.0|         129|         129|            1.0|         0.81|        6.5|  0.5|    0.5|      0.71|         0.0|                  0.3|        8.51|         1.0|                 0.0|       green|
|       2|                 N|       1.0|          75|          42|            3.0|  